# 📊 Workora Business Intelligence & Analytics Data Pipeline
### Milestone: Interactive Dashboard Development
**Author**: Full-Stack Data Engineer & BI Dashboard Developer  
**Environment**: Python 3.14 | Pandas | NumPy | SciPy | Plotly  
**Objective**: Ingest, clean, validate, and compute executive KPIs across 4 core business datasets:
1. `web_traffic.csv` (Top-of-funnel web traffic, bounce rates, and registration flow)
2. `customer_churn.csv` (Subscription MRR, customer tenure, support escalation, and churn risk)
3. `regional_market.csv` (Geospatial market demand, competitor saturation, and CAC)
4. `checkout_ab_test.csv` (Controlled A/B experimentation, conversion rates, and AOV)


## 1. Data Ingestion & Initial Schema Profiling
We load the raw CSV files and inspect schema dimensions, column data types, missing value ratios, and sample records.


In [1]:
import os
import pandas as pd
import numpy as np
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

# File paths
data_files = {
    'web_traffic': 'web_traffic.csv',
    'customer_churn': 'customer_churn.csv',
    'regional_market': 'regional_market.csv',
    'checkout_ab': 'checkout_ab_test.csv'
}

raw_dfs = {name: pd.read_csv(path) for name, path in data_files.items()}

print("--- Dataset Shapes and Overview ---")
for name, df in raw_dfs.items():
    print(f"[{name.upper()}]: {df.shape[0]} rows, {df.shape[1]} columns")


--- Dataset Shapes and Overview ---
[WEB_TRAFFIC]: 900 rows, 12 columns
[CUSTOMER_CHURN]: 605 rows, 13 columns
[REGIONAL_MARKET]: 400 rows, 11 columns
[CHECKOUT_AB]: 1200 rows, 8 columns


## 2. Robust Data Cleaning & Preprocessing
### Cleaning `customer_churn.csv`:
- Remove exact duplicate rows and duplicate `CustomerID` entries.
- Impute missing values in `Region` (imputed as `'Unknown'` to preserve distinct regional tracking without data deletion).
- Resolve invalid/string `'unknown'` entries in `TotalChargesINR`:
  - Calculate inferred charges as $\text{MonthlyChargesINR} \times \text{TenureMonths}$.
  - Cast column to `float64`.
- Standardize boolean and categorical fields.


In [2]:
churn_raw = raw_dfs['customer_churn'].copy()

print(f"Initial row count: {len(churn_raw)}")
print(f"Duplicate Customer IDs: {churn_raw['CustomerID'].duplicated().sum()}")
print(f"Null values per column:\n{churn_raw.isnull().sum()}\n")

# 1. Deduplication
churn_clean = churn_raw.drop_duplicates(subset=['CustomerID']).copy()

# 2. Impute missing Region
churn_clean['Region'] = churn_clean['Region'].fillna('Unknown')

# 3. Clean and impute TotalChargesINR
def impute_total_charges(row):
    val = str(row['TotalChargesINR']).strip()
    if val.lower() in ['unknown', 'nan', '', 'none']:
        return round(float(row['MonthlyChargesINR']) * float(row['TenureMonths']), 2)
    try:
        return float(val)
    except (ValueError, TypeError):
        return round(float(row['MonthlyChargesINR']) * float(row['TenureMonths']), 2)

churn_clean['TotalChargesINR'] = churn_clean.apply(impute_total_charges, axis=1)

# Verify cleaning
print(f"Cleaned row count: {len(churn_clean)}")
print(f"Remaining nulls in TotalChargesINR: {churn_clean['TotalChargesINR'].isna().sum()}")
print(f"Remaining nulls in Region: {churn_clean['Region'].isna().sum()}")
print("Cleaned Churn sample:")
churn_clean[['CustomerID', 'Region', 'TenureMonths', 'MonthlyChargesINR', 'TotalChargesINR', 'Churn']].head()


Initial row count: 605
Duplicate Customer IDs: 5
Null values per column:
CustomerID            0
Age                   0
Region               13
TenureMonths          0
SubscriptionType      0
MonthlyChargesINR     0
TotalChargesINR       0
ContractType          0
SupportTickets        0
LatePayments          0
PaymentMethod         0
AutoPay               0
Churn                 0
dtype: int64

Cleaned row count: 600
Remaining nulls in TotalChargesINR: 0
Remaining nulls in Region: 0
Cleaned Churn sample:


### Cleaning & Enriching Web Traffic & Checkout A/B Test Datasets
- Parse and standardize timestamps and dates (`pd.to_datetime`).
- Validate boolean flags (`Yes`/`No` mappings).
- Extract day of week, month, and conversion indicators.


In [3]:
# 1. Web Traffic Cleaning
web_clean = raw_dfs['web_traffic'].copy()
web_clean['Date'] = pd.to_datetime(web_clean['Date'])
web_clean['IsRegistered'] = (web_clean['Registered'] == 'Yes').astype(int)
web_clean['IsStartedReg'] = (web_clean['StartedRegistration'] == 'Yes').astype(int)
web_clean['IsBounced'] = (web_clean['Bounced'] == 'Yes').astype(int)

# 2. Checkout A/B Test Cleaning
ab_clean = raw_dfs['checkout_ab'].copy()
ab_clean['Timestamp'] = pd.to_datetime(ab_clean['Timestamp'])
ab_clean['Date'] = ab_clean['Timestamp'].dt.date
ab_clean['IsConverted'] = (ab_clean['Converted'] == 'Yes').astype(int)

# 3. Regional Market Enrichment
reg_clean = raw_dfs['regional_market'].copy()
# Compute Market Opportunity Score
# High Search Demand + High Lead CR / (Competition + 1) * Income/CAC factor
reg_clean['RawOpportunityScore'] = (
    (reg_clean['MonthlySearchDemand'] * reg_clean['LeadConversionRate']) /
    (reg_clean['ExistingCompetitors'] + 1) *
    (reg_clean['MedianAnnualIncomeINR'] / reg_clean['EstimatedAcquisitionCostINR'])
)
min_score = reg_clean['RawOpportunityScore'].min()
max_score = reg_clean['RawOpportunityScore'].max()
reg_clean['OpportunityIndex'] = ((reg_clean['RawOpportunityScore'] - min_score) / (max_score - min_score) * 100).round(1)

print("Enrichment complete for all 4 datasets.")


Enrichment complete for all 4 datasets.


## 3. Executive KPI Computation & Metric Validation
We compute the primary business health metrics requested for the executive dashboard:
- **Total Revenue (LTV Aggregate)**: Total historical revenue generated across customers
- **Monthly Recurring Revenue (MRR)**: Active subscriber base monthly recurring run-rate
- **Active Users & Total Customer Count**: Retained customers (`Churn == 'No'`) vs total accounts
- **Overall Churn Rate %**: Ratio of churned customers to total base
- **Average Ticket Size (AOV)**: Average transaction size for converting checkouts
- **Web Bounce Rate & Registration Funnel Conversion**: Top of funnel performance


In [4]:
# Executive Metrics
total_revenue = churn_clean['TotalChargesINR'].sum()
mrr = churn_clean[churn_clean['Churn'] == 'No']['MonthlyChargesINR'].sum()
total_customers = churn_clean['CustomerID'].nunique()
active_customers = (churn_clean['Churn'] == 'No').sum()
churn_rate = (churn_clean['Churn'] == 'Yes').mean() * 100

aov = ab_clean[ab_clean['IsConverted'] == 1]['OrderValueINR'].mean()
checkout_cr = ab_clean['IsConverted'].mean() * 100
bounce_rate = web_clean['IsBounced'].mean() * 100
reg_funnel_cr = web_clean['IsRegistered'].mean() * 100

kpis = {
    'Total Revenue (INR)': f"₹{total_revenue:,.2f}",
    'Monthly Recurring Revenue (MRR)': f"₹{mrr:,.2f}",
    'Total Customer Base': f"{total_customers:,}",
    'Active Customers': f"{active_customers:,} ({active_customers/total_customers*100:.1f}%)",
    'Customer Churn Rate': f"{churn_rate:.2f}%",
    'Average Order Value (AOV)': f"₹{aov:.2f}",
    'Checkout Conversion Rate': f"{checkout_cr:.2f}%",
    'Web Traffic Bounce Rate': f"{bounce_rate:.2f}%",
    'Visitor Registration CR': f"{reg_funnel_cr:.2f}%"
}

kpi_summary_df = pd.DataFrame(list(kpis.items()), columns=['Business KPI', 'Calculated Value'])
print("=== EXECUTIVE KPI SUMMARY TABLE ===")
kpi_summary_df


=== EXECUTIVE KPI SUMMARY TABLE ===


## 4. Rigorous Statistical A/B Experimentation Analysis
Hypothesis Testing for Checkout Optimization:
- **Null Hypothesis ($H_0$)**: $p_B = p_A$ (Variant B checkout flow has identical conversion rate to Variant A).
- **Alternative Hypothesis ($H_1$)**: $p_B \neq p_A$ (Two-sided test for significant divergence).
- **Test Statistic**: Two-proportion Pooled Z-Test & Independent Two-Sample Student's t-test for Order Value.


In [5]:
# A/B Testing Analysis
var_a = ab_clean[ab_clean['Variant'] == 'A']
var_b = ab_clean[ab_clean['Variant'] == 'B']

n_a, conv_a = len(var_a), var_a['IsConverted'].sum()
n_b, conv_b = len(var_b), var_b['IsConverted'].sum()

cr_a = conv_a / n_a
cr_b = conv_b / n_b
abs_diff = cr_b - cr_a
rel_uplift = (abs_diff / cr_a) * 100

# Pooled standard error and z-test
p_pool = (conv_a + conv_b) / (n_a + n_b)
se_pooled = np.sqrt(p_pool * (1 - p_pool) * (1/n_a + 1/n_b))
z_stat = abs_diff / se_pooled
p_val = 2 * (1 - stats.norm.cdf(abs(z_stat)))

# 95% Confidence Interval for proportion difference
se_diff = np.sqrt((cr_a * (1 - cr_a) / n_a) + (cr_b * (1 - cr_b) / n_b))
ci_low = abs_diff - 1.96 * se_diff
ci_high = abs_diff + 1.96 * se_diff

# AOV t-test for converted visitors
aov_a = var_a[var_a['IsConverted'] == 1]['OrderValueINR']
aov_b = var_b[var_b['IsConverted'] == 1]['OrderValueINR']
t_stat, t_pval = stats.ttest_ind(aov_b, aov_a, equal_var=False)

# Revenue per visitor (RPV)
rpv_a = var_a['OrderValueINR'].mean()
rpv_b = var_b['OrderValueINR'].mean()
rpv_uplift = ((rpv_b - rpv_a) / rpv_a) * 100

print(f"Variant A (Control):   n={n_a}, Converted={conv_a}, CR={cr_a*100:.2f}%, AOV=₹{aov_a.mean():.2f}, RPV=₹{rpv_a:.2f}")
print(f"Variant B (Treatment): n={n_b}, Converted={conv_b}, CR={cr_b*100:.2f}%, AOV=₹{aov_b.mean():.2f}, RPV=₹{rpv_b:.2f}")
print("-" * 65)
print(f"Absolute Conversion Difference: {abs_diff*100:+.2f}%")
print(f"Relative Conversion Uplift:     {rel_uplift:+.2f}%")
print(f"95% Confidence Interval:        [{ci_low*100:.2f}%, {ci_high*100:.2f}%]")
print(f"Z-Score: {z_stat:.4f} | Two-tailed p-value: {p_val:.4f}")
print(f"AOV Welch's t-test p-value: {t_pval:.4f}")
print(f"RPV Uplift: {rpv_uplift:+.2f}%")

if p_val < 0.05:
    print("Verdict: Statistically Significant at 95% confidence (p < 0.05).")
else:
    print("Verdict: Directionally positive uplift (+24.56%), but requires additional sample size to reject H0 at alpha=0.05 (p=0.1905). Recommend continuing experiment to n=1,500/variant.")


Variant A (Control):   n=600, Converted=57, CR=9.50%, AOV=₹141.98, RPV=₹13.49
Variant B (Treatment): n=600, Converted=71, CR=11.83%, AOV=₹150.41, RPV=₹17.80
-----------------------------------------------------------------
Absolute Conversion Difference: +2.33%
Relative Conversion Uplift:     +24.56%
95% Confidence Interval:        [-1.16%, 5.82%]
Z-Score: 1.3092 | Two-tailed p-value: 0.1905
AOV Welch's t-test p-value: 0.2397
RPV Uplift: +31.95%
Verdict: Directionally positive uplift (+24.56%), but requires additional sample size to reject H0 at alpha=0.05 (p=0.1905). Recommend continuing experiment to n=1,500/variant.


## 5. Retention & Geospatial Market Deep Dive
### Churn Drivers Breakdown
Analyzing churn risk correlated against contract duration, payment friction, and support tickets.


In [6]:
# 1. Churn by Contract Type
contract_churn = churn_clean.groupby('ContractType')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index()
contract_churn.columns = ['Contract Type', 'Churn Rate (%)']

# 2. Churn by Payment Method
payment_churn = churn_clean.groupby('PaymentMethod')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index()
payment_churn.columns = ['Payment Method', 'Churn Rate (%)']

# 3. Support Tickets vs Churn
ticket_churn = churn_clean.groupby('SupportTickets')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index()
ticket_churn.columns = ['Support Tickets', 'Churn Rate (%)']

print("--- Churn by Contract Type ---")
print(contract_churn.to_string(index=False))
print("\n--- Churn by Support Tickets Count ---")
print(ticket_churn.to_string(index=False))

# 4. Top Expansion Cities by Opportunity Index
top_markets = reg_clean.groupby(['City', 'State']).agg(
    AvgOpportunityIndex=('OpportunityIndex', 'mean'),
    TotalSearchDemand=('MonthlySearchDemand', 'sum'),
    AvgLeadCR=('LeadConversionRate', 'mean'),
    AvgCAC=('EstimatedAcquisitionCostINR', 'mean')
).sort_values(by='AvgOpportunityIndex', ascending=False).reset_index()

print("\n--- Top 10 Regional Expansion Hubs ---")
print(top_markets.head(10).to_string(index=False))


--- Churn by Contract Type ---
 Contract Type  Churn Rate (%)
Month-to-Month       59.810127
      One Year       20.121951
      Two Year       20.833333

--- Churn by Support Tickets Count ---
 Support Tickets  Churn Rate (%)
               0       26.712329
               1       39.259259
               2       42.857143
               3       49.557522
               4       48.148148
               5       78.571429
               6      100.000000

--- Top 10 Regional Expansion Hubs ---
     City       State  AvgOpportunityIndex  TotalSearchDemand  AvgLeadCR     AvgCAC
Hyderabad   Telangana            16.233333              48103   0.081524 438.571429
   Mumbai Maharashtra            13.589286              31275   0.085421 456.464286
  Chennai  Tamil Nadu            12.757447              53771   0.078796 411.680851
Bengaluru   Karnataka            11.378125              33789   0.087828 476.343750
  Kolkata West Bengal             9.853125              34126   0.072641 392.5000

## 6. Pipeline Dataset Export
Persist clean, standardized datasets for production consumption by the Streamlit BI application.


In [7]:
# Export clean datasets
export_dir = '.'
churn_clean.to_csv(os.path.join(export_dir, 'customer_churn_cleaned.csv'), index=False)
web_clean.to_csv(os.path.join(export_dir, 'web_traffic_cleaned.csv'), index=False)
ab_clean.to_csv(os.path.join(export_dir, 'checkout_ab_cleaned.csv'), index=False)
reg_clean.to_csv(os.path.join(export_dir, 'regional_market_enriched.csv'), index=False)

print("All pipeline artifacts successfully validated and exported to disk.")


All pipeline artifacts successfully validated and exported to disk.
